# 面试问题：INT8 量化怎样从零实现，PTQ 与 QAT 如何取舍？

**一句话回答**：用 scale 与 zero-point 把浮点范围映射到整数网格，权重常用对称 per-output-channel，激活常用动态或校准后的 per-tensor。整数 GEMM 在 int32 中累加再按 scale 反量化；PTQ 依赖代表性校准与敏感度分析，QAT 用 fake quant + STE 让模型适应舍入和饱和误差。

本 Notebook 用 NumPy 实现 qparam、量化/反量化与整数 Linear，并用 PyTorch 自定义 autograd 实现 QAT 的 fake quant。面试中还应说明：离线文件变小不等于线上加速，最终收益取决于目标硬件是否有对应 INT8 kernel、算子能否融合、内存带宽是否成为瓶颈，以及量化与反量化边界是否过多。

In [ ]:
import hashlib, json, math
import numpy as np
import torch
from torch import nn

SEED101=10101; rng101=np.random.default_rng(SEED101); torch.manual_seed(SEED101)
values101=rng101.normal(size=2048).astype(np.float32)
assert values101.dtype==np.float32 and np.isfinite(values101).all()
assert len(values101)==2048
assert SEED101==10101

## 1. 对称 INT8：零点固定为 0

对称量化取 `scale=max(|x|)/127`，`q=clip(round(x/scale),-127,127)`，反量化为 `q·scale`。不用 -128 能让正负范围对称。它适合近似零中心的权重，计算和 zero-point 修正都更简单。

In [ ]:
def symmetric_qparams101(x,axis=None,eps=1e-12):
    m=np.max(np.abs(x),axis=axis,keepdims=axis is not None); return np.maximum(m/127.,eps)
def quant_sym101(x,scale): return np.clip(np.rint(np.asarray(x)/scale),-127,127).astype(np.int8)
def dequant_sym101(q,scale): return q.astype(np.float32)*scale
scale101=symmetric_qparams101(values101); q101=quant_sym101(values101,scale101); dq101=dequant_sym101(q101,scale101)
assert q101.dtype==np.int8 and q101.min()>=-127 and q101.max()<=127
assert float(np.max(np.abs(values101-dq101)))<=float(scale101)/2+1e-6
assert math.isclose(float(dequant_sym101(quant_sym101([0.],scale101),scale101)[0]),0.)

## 2. 非对称 affine：用 zero-point 表示偏移

对 ReLU 激活可映射到 uint8 `[0,255]`：`scale=(max-min)/255`，`zero=round(-min/scale)`。非对称能更充分利用偏移分布的码字，但整数 GEMM 需要减 zero-point，并保证 zero 被准确表示。

In [ ]:
def affine_qparams101(x,qmin=0,qmax=255):
    lo,hi=float(np.min(x)),float(np.max(x)); scale=max((hi-lo)/(qmax-qmin),1e-12); zero=int(np.clip(round(qmin-lo/scale),qmin,qmax)); return scale,zero
def quant_affine101(x,scale,zero): return np.clip(np.rint(np.asarray(x)/scale)+zero,0,255).astype(np.uint8)
act101=np.maximum(values101[:512]+1.5,0); sa101,za101=affine_qparams101(act101); qa101=quant_affine101(act101,sa101,za101); dqa101=(qa101.astype(np.float32)-za101)*sa101
assert qa101.dtype==np.uint8 and 0<=za101<=255
assert np.max(np.abs(act101-dqa101))<=sa101/2+1e-6
assert quant_affine101([0.],sa101,za101)[0]==za101

## 3. 权重为何常用 per-channel

不同输出通道的动态范围可能差几个数量级。per-tensor 的大通道会迫使小通道使用粗网格；沿输入维求每个输出通道的 scale，通常显著降低权重误差，代价是每个通道保存一个 scale。

In [ ]:
w101=np.vstack([rng101.normal(0,.02,64),rng101.normal(0,.5,64),rng101.normal(0,4.,64)]).astype(np.float32)
st101=symmetric_qparams101(w101); wt101=dequant_sym101(quant_sym101(w101,st101),st101); sc101=symmetric_qparams101(w101,axis=1); wc101=dequant_sym101(quant_sym101(w101,sc101),sc101)
mse_t101=((w101-wt101)**2).mean(axis=1); mse_c101=((w101-wc101)**2).mean(axis=1)
assert sc101.shape==(3,1)
assert mse_c101[0]<mse_t101[0]*.01
assert mse_c101.mean()<mse_t101.mean()

## 4. INT8 Linear：int32 累加后再缩放

对称情形中 `X≈sₓQₓ, W_o≈s_wo Q_wo`，所以 `Y_o≈(Qₓ·Q_wo)·sₓs_wo+b_o`。乘加必须用 int32，避免长向量在 int8 中溢出。真实 kernel 还会融合 requantize 与下一层激活。

In [ ]:
xlin101=rng101.normal(size=(8,16)).astype(np.float32); wlin101=rng101.normal(size=(5,16)).astype(np.float32); blin101=rng101.normal(size=5).astype(np.float32)
sx101=symmetric_qparams101(xlin101); sw101=symmetric_qparams101(wlin101,axis=1); qx101=quant_sym101(xlin101,sx101); qw101=quant_sym101(wlin101,sw101)
accum101=qx101.astype(np.int32)@qw101.astype(np.int32).T; yint101=accum101.astype(np.float32)*(float(sx101)*sw101.reshape(1,-1))+blin101; yfp101=xlin101@wlin101.T+blin101
assert accum101.dtype==np.int32 and yint101.shape==(8,5)
assert np.mean(np.abs(yint101-yfp101))<.08
assert np.isfinite(yint101).all()

## 5. Bias 可以在 accumulator 尺度中量化

输出通道 bias 的尺度是 `sₓ·s_w[o]`，量化为 int32 后可在反量化前直接加到 accumulator。若输入 scale 动态变化，bias 整数值也需相应处理；不少部署后端保留 FP32 bias 并融合计算。

In [ ]:
bias_scale101=float(sx101)*sw101.reshape(-1); qbias101=np.rint(blin101/bias_scale101).astype(np.int32); ybias101=(accum101+qbias101.reshape(1,-1)).astype(np.float32)*bias_scale101.reshape(1,-1)
no_bias101=accum101.astype(np.float32)*bias_scale101.reshape(1,-1)
assert qbias101.dtype==np.int32 and qbias101.shape==(5,)
assert np.max(np.abs((ybias101-no_bias101)-blin101))<=float(bias_scale101.max())/2+1e-6
assert np.mean(np.abs(ybias101-yfp101))<.08

## 6. PTQ 校准：最大范围不一定最小任务误差

极少数 outlier 会让 min-max scale 过大，使主体样本量化很粗。percentile/MSE/KL 校准允许少量饱和换取主体分辨率。校准集必须代表线上分布，并按算子记录范围、饱和率和误差；数据漂移后需重新校准。

In [ ]:
calib101=np.concatenate([rng101.normal(0,1,10000),np.array([40.])]).astype(np.float32); bulk101=np.abs(calib101)<3
smax101=symmetric_qparams101(calib101); spct101=max(np.percentile(np.abs(calib101),99.9)/127,1e-12)
dqmax101=dequant_sym101(quant_sym101(calib101,smax101),smax101); dqpct101=dequant_sym101(quant_sym101(calib101,spct101),spct101)
assert spct101<smax101
assert np.mean((dqpct101[bulk101]-calib101[bulk101])**2)<np.mean((dqmax101[bulk101]-calib101[bulk101])**2)
assert abs(float(dqpct101[-1]))<abs(float(calib101[-1]))

## 7. QAT：forward 模拟量化，backward 用 STE

fake quant 的 forward 执行 round/clip/dequant，参数仍是浮点；round 几乎处处导数为 0，因此 Straight-Through Estimator 在有效范围内近似传递梯度。下面显式实现 autograd Function，证明量化噪声存在且梯度仍能训练。

In [ ]:
class FakeQuantSTE101(torch.autograd.Function):
    @staticmethod
    def forward(ctx,x,scale): return (torch.clamp(torch.round(x/scale),-127,127)*scale)
    @staticmethod
    def backward(ctx,grad_output): return grad_output,None
class QLinear101(nn.Module):
    def __init__(self,inn,out): super().__init__(); self.weight=nn.Parameter(torch.randn(out,inn)*.2); self.bias=nn.Parameter(torch.zeros(out))
    def forward(self,x):
        sx=x.detach().abs().max().clamp_min(1e-8)/127; sw=self.weight.detach().abs().amax(dim=1,keepdim=True).clamp_min(1e-8)/127
        return FakeQuantSTE101.apply(x,sx)@FakeQuantSTE101.apply(self.weight,sw).t()+self.bias
qlayer101=QLinear101(4,3); qx_t101=torch.randn(6,4,requires_grad=True); qout101=qlayer101(qx_t101); qout101.square().mean().backward()
assert qout101.shape==(6,3)
assert qlayer101.weight.grad is not None and qx_t101.grad is not None
assert torch.isfinite(qlayer101.weight.grad).all()

## 8. 敏感度分析与部署合同

不是所有层都必须 INT8。常见策略是逐层量化，观察输出 SQNR 与验证指标，对首尾层、Softmax、LayerNorm 或异常通道保留更高精度。最终 manifest 固定 scheme、axis、rounding、校准数据版本和后端；否则离线模拟与线上 kernel 可能不一致。

In [ ]:
def sqnr101(x,xq):
    noise=np.sum((x-xq)**2); return float("inf") if noise==0 else float(10*np.log10(np.sum(x*x)/noise))
sqnr_tensor101=sqnr101(w101,wt101); sqnr_channel101=sqnr101(w101,wc101)
manifest101={"dtype":"int8","weight":{"scheme":"symmetric","granularity":"per_output_channel","axis":0},"activation":{"scheme":"affine","calibration":"percentile"},"accumulator":"int32","rounding":"nearest_even"}; digest101=hashlib.sha256(json.dumps(manifest101,sort_keys=True).encode()).hexdigest()
assert sqnr_channel101>sqnr_tensor101
assert manifest101["weight"]["axis"]==0 and manifest101["accumulator"]=="int32"
assert len(digest101)==64 and math.isfinite(sqnr_channel101)

## 面试总结

建议按 **qparam/舍入/饱和 → 对称与 affine → per-channel → int32 GEMM/bias → PTQ 校准 → QAT+STE → 敏感层回退 → 后端 manifest** 回答。量化的目标不是“权重变成 int8”，而是部署 kernel、精度、延迟、模型大小和数据漂移共同满足合同。

延伸阅读：[PyTorch Quantization](https://pytorch.org/docs/stable/quantization.html)、[TensorRT INT8 Calibration](https://docs.nvidia.com/deeplearning/tensorrt/developer-guide/index.html#working-with-int8)、[Quantization and Training of Neural Networks](https://arxiv.org/abs/1712.05877)。